In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)

import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier
from IPython.display import display

In [2]:
warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================
DATA_PATH   = "processed/chennai_aqi_era5_hourly.parquet"
OUT_DIR     = "huber_processed_leakfree"
SPLIT_TIME  = pd.Timestamp("2025-01-01 00:00:00")
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4

# Tuning: reduce these for quick debugging, restore for final paper runs
RUN_TUNED   = True
JSO_POP     = 12
JSO_ITERS   = 15
RANDOM_SEED = 1


In [3]:
# =========================================================
# HELPERS
# =========================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.arctan2(y, x)

def flatten_window_per_node(X):
    S, Lx, N, F = X.shape
    return X.transpose(0, 2, 1, 3).reshape(S * N, Lx * F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90, 95, 99)):
    rows = []
    for p in percentiles:
        thr = np.percentile(y_true, p)
        idx = y_true >= thr
        if idx.sum() == 0:
            rows.append((p, thr, np.nan, np.nan, 0))
        else:
            rows.append((
                p,
                thr,
                mean_absolute_error(y_true[idx], y_pred[idx]),
                np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])),
                int(idx.sum())
            ))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mfb = np.mean(2.0 * (y_pred - y_true) / (y_pred + y_true + 1e-8))
    nmse = np.sum((y_pred - y_true) ** 2) / (np.sum(y_pred * y_true) + 1e-8)
    return mfb, nmse

def build_fill_values_from_train_timeline(X_train_timeline):
    # X_train_timeline: [T_train, N, F]
    fill_values = np.nanmedian(X_train_timeline, axis=0)  # [N, F]
    global_fill = np.nanmedian(X_train_timeline.reshape(-1, X_train_timeline.shape[-1]), axis=0)  # [F]
    global_fill = np.where(np.isnan(global_fill), 0.0, global_fill)
    for n in range(fill_values.shape[0]):
        for f in range(fill_values.shape[1]):
            if np.isnan(fill_values[n, f]):
                fill_values[n, f] = global_fill[f]
    fill_values = np.where(np.isnan(fill_values), 0.0, fill_values).astype(np.float32)
    return fill_values

def impute_windows(X, fill_values):
    # Strictly leakage-safe: operate within each window only, then fall back to TRAIN-only fill_values.
    X_imp = X.copy().astype(np.float32)
    S, Lx, N, F = X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                series = pd.Series(X_imp[s, :, n, f], dtype="float32")
                series = series.ffill().bfill()  # only inside the current historical window
                arr = series.to_numpy(dtype=np.float32)
                if np.isnan(arr).any():
                    arr = np.where(np.isnan(arr), fill_values[n, f], arr)
                X_imp[s, :, n, f] = arr
    return X_imp

def build_nodes_edges(df, station_ids, k=4):
    coords = (
        df[["station_id", "lat", "lon"]]
        .drop_duplicates()
        .assign(station_id=lambda d: d["station_id"].astype(str))
    )
    nodes = coords[coords["station_id"].isin(station_ids)].copy()
    nodes = nodes.set_index("station_id").loc[station_ids].reset_index()
    nodes["node_id"] = np.arange(len(nodes))

    coords_arr = nodes[["lat", "lon"]].to_numpy(dtype=float)
    N = len(nodes)

    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords_arr[i, 0], coords_arr[i, 1], coords_arr[:, 0], coords_arr[:, 1])

    sigma = np.median(D[D > 0])
    edges = []
    kk = min(k, N - 1)
    for i in range(N):
        nn = np.argsort(D[i])[1:kk + 1]
        for j in nn:
            w = np.exp(-(D[i, j] ** 2) / (2.0 * sigma ** 2))
            edges.append((i, j, float(w), float(D[i, j])))

    edges_df = pd.DataFrame(edges, columns=["src", "dst", "w_dist", "dist_km"])
    return nodes, edges_df

def build_static_adj_from_nodes(nodes_df, k=4, sigma_km=None, self_loops=False, row_normalize=True):
    coords = nodes_df[["lat", "lon"]].to_numpy(dtype=float)
    N = coords.shape[0]
    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords[i, 0], coords[i, 1], coords[:, 0], coords[:, 1])

    if sigma_km is None:
        sigma_km = np.median(D[D > 0])

    A = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k + 1, N)]
        for j in nn:
            A[i, j] = np.float32(np.exp(-(D[i, j] ** 2) / (2.0 * sigma_km ** 2)))

    if self_loops:
        np.fill_diagonal(A, 1.0)

    if row_normalize:
        rs = A.sum(axis=1, keepdims=True)
        A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)

    return A

def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05):
    theta_w = np.arctan2(v_t[ctx["src"]], u_t[ctx["src"]])  # wind direction at src nodes
    align = np.cos(theta_w - ctx["edge_bearing"])           # directional alignment with edge
    gate = 1.0 / (1.0 + np.exp(-alpha * align))
    gate = eps + (1.0 - eps) * gate
    w_dyn = ctx["w_dist"] * gate

    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[ctx["src"], ctx["dst"]] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)
    return A

def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        A = build_dynamic_adj(
            x_now[:, ctx["u_idx"]],
            x_now[:, ctx["v_idx"]],
            ctx,
            alpha=alpha,
            eps=eps
        )
        agg = A @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z

def make_graph_features_static(X, A_static, tau=0):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        agg = A_static @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z

def summarize_regression(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred))
    }

def train_val_split_time_order(X_train_imp, Y_train, M_train, target_times_train, frac=0.2):
    n = X_train_imp.shape[0]
    val_size = max(1, int(np.ceil(frac * n)))
    idx_train = np.arange(0, n - val_size)
    idx_val   = np.arange(n - val_size, n)
    train_pack = {
        "X": X_train_imp[idx_train],
        "Y": Y_train[idx_train],
        "M": M_train[idx_train],
        "times": target_times_train[idx_train]
    }
    val_pack = {
        "X": X_train_imp[idx_val],
        "Y": Y_train[idx_val],
        "M": M_train[idx_val],
        "times": target_times_train[idx_val]
    }
    return train_pack, val_pack

def fit_flat_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test):
    Xtr = flatten_window_per_node(X_train_imp)
    Xte = flatten_window_per_node(X_test_imp)
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_static_graph_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, A_static, tau):
    Ztr = make_graph_features_static(X_train_imp, A_static, tau=tau)
    Zte = make_graph_features_static(X_test_imp, A_static, tau=tau)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_huber_graph(X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4):
    Ztr = make_graph_features_dynamic(X_train_imp, ctx, tau=tau, alpha=alpha, eps=eps)
    Zte = make_graph_features_dynamic(X_test_imp, ctx, tau=tau, alpha=alpha, eps=eps)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    huber = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, alpha=huber_alpha, max_iter=500))
    ])
    huber.fit(Xtr[mtr], ytr[mtr])

    pred_train_all = huber.predict(Xtr)
    pred_test_all  = huber.predict(Xte)

    return {
        "y_true": yte[mte],
        "y_pred": pred_test_all[mte],
        "pred_train_all": pred_train_all,
        "pred_test_all": pred_test_all,
        "y_train_all": ytr,
        "y_test_all": yte,
        "mask_train": mtr,
        "mask_test": mte,
        "model": huber
    }

def fit_huber_hybrid(
    X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
    ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4,
    xgb_params=None, num_boost_round=400, seed=42,
    w_mid=2.0, w_danger=5.0, w_tail=10.0
):
    base = fit_huber_graph(
        X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
        ctx=ctx, tau=tau, alpha=alpha, eps=eps, huber_alpha=huber_alpha
    )

    ytr_all = base["y_train_all"]
    pred_tr_all = base["pred_train_all"]
    mtr = base["mask_train"]
    mte = base["mask_test"]

    res_train = ytr_all - pred_tr_all
    Xgb_train = np.asarray(flatten_window_per_node(X_train_imp), dtype=np.float32)
    Xgb_test  = np.asarray(flatten_window_per_node(X_test_imp),  dtype=np.float32)

    t90 = np.percentile(ytr_all[mtr], 90)
    t95 = np.percentile(ytr_all[mtr], 95)
    t99 = np.percentile(ytr_all[mtr], 99)

    weights = np.ones_like(ytr_all[mtr], dtype=np.float32)
    weights[ytr_all[mtr] >= t90] = w_mid
    weights[ytr_all[mtr] >= t95] = w_danger
    weights[ytr_all[mtr] >= t99] = w_tail

    dtrain = xgb.DMatrix(Xgb_train[mtr], label=res_train[mtr], weight=weights)

    if xgb_params is None:
        xgb_params = {
            "objective": "reg:pseudohubererror",
            "max_depth": 6,
            "eta": 0.05,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "lambda": 1.0,
            "tree_method": "hist",
            "seed": seed,
            "verbosity": 0,
        }

    booster = xgb.train(xgb_params, dtrain, num_boost_round=int(num_boost_round))

    res_test_all = booster.predict(xgb.DMatrix(Xgb_test))
    final_test_all = base["pred_test_all"].copy()
    final_test_all[mte] = final_test_all[mte] + res_test_all[mte]

    return {
        "y_true": base["y_test_all"][mte],
        "y_pred_graph": base["pred_test_all"][mte],
        "y_pred_final": final_test_all[mte],
        "gmodel": base["model"],
        "hmodel": booster
    }

def choose_best_tau_dynamic(train_pack, val_pack, ctx, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        res = fit_huber_graph(
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            ctx=ctx, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(train_pack, val_pack, A_static, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("huber", HuberRegressor(epsilon=1.35, max_iter=500))
        ])
        res = fit_static_graph_regressor(
            model,
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            A_static=A_static, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, train_pack, val_pack, ctx):
    alpha, eps, log_huber_alpha, max_depth, eta, subsample, colsample, reg_lambda, nround, w_mid, w_danger, w_tail = params
    huber_alpha = 10 ** float(log_huber_alpha)

    hybrid = fit_huber_hybrid(
        train_pack["X"], train_pack["Y"], train_pack["M"],
        val_pack["X"],   val_pack["Y"],   val_pack["M"],
        ctx=ctx,
        tau=tau,
        alpha=float(alpha),
        eps=float(eps),
        huber_alpha=float(huber_alpha),
        xgb_params={
            "objective": "reg:pseudohubererror",
            "max_depth": int(round(max_depth)),
            "eta": float(eta),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample),
            "lambda": float(reg_lambda),
            "tree_method": "hist",
            "seed": RANDOM_SEED,
            "verbosity": 0,
        },
        num_boost_round=int(round(nround)),
        seed=RANDOM_SEED,
        w_mid=float(w_mid),
        w_danger=float(w_danger),
        w_tail=float(w_tail)
    )

    y_true = hybrid["y_true"]
    y_pred = hybrid["y_pred_final"]
    mae = mean_absolute_error(y_true, y_pred)

    thr99 = np.percentile(y_true, 99)
    idx_tail = y_true >= thr99
    if idx_tail.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx_tail], y_pred[idx_tail])
        return mae + (10.0 * abs(mfb_tail))
    return mae

def sample_population(n, bounds):
    pop = np.random.rand(n, bounds.shape[0])
    return bounds[:, 0] + pop * (bounds[:, 1] - bounds[:, 0])

def clip_params(x, bounds):
    return np.minimum(np.maximum(x, bounds[:, 0]), bounds[:, 1])

def jso_optimize(train_pack, val_pack, ctx, tau, bounds, n_pop=12, iters=15, seed=1):
    np.random.seed(seed)
    dim = bounds.shape[0]
    pop = sample_population(n_pop, bounds)
    fitness = np.array([eval_hybrid_params(p, tau, train_pack, val_pack, ctx) for p in pop], dtype=float)

    best_idx = np.argmin(fitness)
    best_p = pop[best_idx].copy()
    best_f = float(fitness[best_idx])
    print(f"    initial best objective={best_f:.4f}")

    for it in range(iters):
        c = (1.0 - it / max(iters, 1))
        new_pop = pop.copy()
        for i in range(n_pop):
            if np.random.rand() < 0.5:
                step = np.random.randn(dim) * c * 0.1
                cand = pop[i] + step + c * (best_p - pop[i]) * np.random.rand(dim)
            else:
                j = np.random.randint(0, n_pop)
                step = (pop[j] - pop[i]) * (np.random.rand(dim) - 0.5) * c
                cand = pop[i] + step
            new_pop[i] = clip_params(cand, bounds)

        new_fit = np.array([eval_hybrid_params(p, tau, train_pack, val_pack, ctx) for p in new_pop], dtype=float)
        improved = new_fit < fitness
        pop[improved] = new_pop[improved]
        fitness[improved] = new_fit[improved]

        best_idx = np.argmin(fitness)
        if fitness[best_idx] < best_f:
            best_f = float(fitness[best_idx])
            best_p = pop[best_idx].copy()

        print(f"    iter {it + 1:02d}/{iters:02d} -> best objective={best_f:.4f}")

    return best_p, best_f

def add_result_row(rows, tail_rows, H, split_name, model_name, y_true, y_pred, extra=None):
    summary = summarize_regression(y_true, y_pred)
    row = {
        "H": H,
        "split": split_name,
        "model": model_name,
        "MAE": summary["MAE"],
        "RMSE": summary["RMSE"],
        "R2": summary["R2"],
        "n_test": int(len(y_true))
    }
    if extra:
        row.update(extra)
    rows.append(row)

    for p, thr, mae, rmse, n_tail in compute_tail_metrics(y_true, y_pred):
        tail_row = {
            "H": H,
            "split": split_name,
            "model": model_name,
            "percentile": p,
            "threshold": float(thr),
            "tail_MAE": float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE": float(rmse) if pd.notna(rmse) else np.nan,
            "n_tail": int(n_tail)
        }
        if extra:
            tail_row.update(extra)
        tail_rows.append(tail_row)


In [4]:
# =========================================================
# 1) BUILD LEAKAGE-FREE RAW TENSOR
# =========================================================
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH).copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["station_id"] = df["station_id"].astype(str)

stations = (
    df[["station_id", "station_name"]]
    .drop_duplicates()
    .sort_values("station_id")
    .reset_index(drop=True)
)
station_ids = stations["station_id"].tolist()
N = len(station_ids)

candidate_features = ["pm25", "temp_2m", "dewpoint_2m", "surface_pressure", "u10", "v10"]
features = [c for c in candidate_features if c in df.columns]
assert "pm25" in features, "pm25 column not found in parquet."
pm25_idx = features.index("pm25")
u_idx = features.index("u10")
v_idx = features.index("v10")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids], names=["timestamp", "station_id"]).to_frame(index=False)

aligned = base.merge(
    df[["timestamp", "station_id"] + features],
    on=["timestamp", "station_id"],
    how="left"
)

X_feat = []
for feat in features:
    mat = aligned.pivot(index="timestamp", columns="station_id", values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))

X_all_raw = np.stack(X_feat, axis=-1).astype(np.float32)   # [T, N, F]

pm25_raw = aligned.pivot(index="timestamp", columns="station_id", values="pm25").reindex(all_times)[station_ids]
pm25_raw_np = pm25_raw.to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(pm25_raw_np)).astype(np.float32)

# Fill values are learned from TRAIN timeline only -> no leakage from future/test period
train_timeline_mask = all_times < SPLIT_TIME
fill_values = build_fill_values_from_train_timeline(X_all_raw[train_timeline_mask])

# =========================================================
# 2) BUILD GRAPH STRUCTURE ONCE
# =========================================================
nodes, edges_df = build_nodes_edges(df, station_ids, k=GRAPH_K)
A_static = build_static_adj_from_nodes(nodes, k=GRAPH_K, sigma_km=None, self_loops=False, row_normalize=True)

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)

src_lat = nodes.loc[src, "lat"].to_numpy()
src_lon = nodes.loc[src, "lon"].to_numpy()
dst_lat = nodes.loc[dst, "lat"].to_numpy()
dst_lon = nodes.loc[dst, "lon"].to_numpy()
edge_bearing = bearing_radians(src_lat, src_lon, dst_lat, dst_lon)

graph_ctx = {
    "src": src,
    "dst": dst,
    "w_dist": w_dist,
    "edge_bearing": edge_bearing,
    "u_idx": u_idx,
    "v_idx": v_idx,
    "N": len(nodes),
}

# =========================================================
# 3) BUILD LEAKAGE-FREE WINDOWS PER H
# =========================================================
all_data = {}

for H in H_LIST:
    Xs, Ys, Ms, ref_times, target_times = [], [], [], [], []

    for t in range(L, len(all_times) - H):
        x_raw = X_all_raw[t - L:t, :, :]     # history only
        y = pm25_raw_np[t + H, :]            # target at t+H
        m = Y_mask_full[t + H, :]

        if m.sum() == 0:
            continue

        Xs.append(x_raw)
        Ys.append(y)
        Ms.append(m)
        ref_times.append(all_times[t])       # forecast issuance/reference time
        target_times.append(all_times[t + H])

    X = np.stack(Xs).astype(np.float32)
    Y = np.stack(Ys).astype(np.float32)
    M = np.stack(Ms).astype(np.float32)
    ref_times = pd.to_datetime(pd.Series(ref_times))
    target_times = pd.to_datetime(pd.Series(target_times))

    train_idx = (target_times < SPLIT_TIME).to_numpy()
    test_idx  = (target_times >= SPLIT_TIME).to_numpy()

    X_train_raw, Y_train, M_train = X[train_idx], Y[train_idx], M[train_idx]
    X_test_raw,  Y_test,  M_test  = X[test_idx],  Y[test_idx],  M[test_idx]

    X_train_imp = impute_windows(X_train_raw, fill_values)
    X_test_imp  = impute_windows(X_test_raw,  fill_values)

    out_h = os.path.join(OUT_DIR, f"H{H}")
    os.makedirs(out_h, exist_ok=True)
    np.save(os.path.join(out_h, "X_train.npy"), X_train_raw)
    np.save(os.path.join(out_h, "Y_train.npy"), Y_train)
    np.save(os.path.join(out_h, "M_train.npy"), M_train)
    np.save(os.path.join(out_h, "X_test.npy"),  X_test_raw)
    np.save(os.path.join(out_h, "Y_test.npy"),  Y_test)
    np.save(os.path.join(out_h, "M_test.npy"),  M_test)
    np.save(os.path.join(out_h, "X_train_imp.npy"), X_train_imp)
    np.save(os.path.join(out_h, "X_test_imp.npy"),  X_test_imp)

    all_data[H] = {
        "X_train": X_train_raw,
        "Y_train": Y_train,
        "M_train": M_train,
        "X_test": X_test_raw,
        "Y_test": Y_test,
        "M_test": M_test,
        "X_train_imp": X_train_imp,
        "X_test_imp": X_test_imp,
        "ref_times_train": ref_times[train_idx].reset_index(drop=True),
        "ref_times_test": ref_times[test_idx].reset_index(drop=True),
        "target_times_train": target_times[train_idx].reset_index(drop=True),
        "target_times_test": target_times[test_idx].reset_index(drop=True),
    }

    print(f"H={H:>2} | train={X_train_imp.shape[0]:>5} | test={X_test_imp.shape[0]:>5} | "
          f"first train target={all_data[H]['target_times_train'].iloc[0]} | "
          f"last train target={all_data[H]['target_times_train'].iloc[-1]} | "
          f"first test target={all_data[H]['target_times_test'].iloc[0]}")

# =========================================================
# 4) RUN ABLATIONS + GRAPH MODELS IN ONE LOOP
#    NOTE:
#    - tau is selected on TRAIN/VAL only -> no test leakage
#    - all models use the same validity masks from M_train / M_test
# =========================================================
results_rows = []
tail_rows = []
tau_rows = []
all_models = {}

bounds = np.array([
    [1.0, 8.0],     # alpha
    [0.01, 0.15],   # eps
    [-6.0, -2.0],   # log10(huber_alpha)
    [3.0, 10.0],    # max_depth
    [0.01, 0.20],   # eta
    [0.60, 1.00],   # subsample
    [0.60, 1.00],   # colsample_bytree
    [0.00, 10.00],  # lambda
    [100.0, 800.0], # num_boost_round
    [1.0, 5.0],     # w_mid
    [1.1, 10.0],    # w_danger
    [2.0, 30.0],    # w_tail
], dtype=float)

for H in H_LIST:
    print("\n" + "=" * 80)
    print(f"RUNNING H={H}")
    print("=" * 80)

    pack = all_data[H]
    X_train_imp = pack["X_train_imp"]
    Y_train = pack["Y_train"]
    M_train = pack["M_train"]
    X_test_imp = pack["X_test_imp"]
    Y_test = pack["Y_test"]
    M_test = pack["M_test"]

    train_pack, val_pack = train_val_split_time_order(
        X_train_imp, Y_train, M_train, pack["target_times_train"].to_numpy(), frac=VALID_FRAC
    )

    # -------------------------
    # Tau selection on validation only
    # -------------------------
    best_tau_dyn, tau_table_dyn = choose_best_tau_dynamic(train_pack, val_pack, graph_ctx, TAU_LIST)
    best_tau_static, tau_table_static = choose_best_tau_static(train_pack, val_pack, A_static, TAU_LIST)

    tau_table_dyn["H"] = H
    tau_table_dyn["family"] = "dynamic"
    tau_table_static["H"] = H
    tau_table_static["family"] = "static"
    tau_rows.extend(tau_table_dyn.to_dict("records"))
    tau_rows.extend(tau_table_static.to_dict("records"))

    print(f"Selected tau (dynamic) = {best_tau_dyn}")
    print(f"Selected tau (static)  = {best_tau_static}")

    all_models[H] = {
        "selected_tau_dynamic": best_tau_dyn,
        "selected_tau_static": best_tau_static
    }

    # -------------------------
    # ABLATIONS: no-graph
    # -------------------------
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0))
    ])
    res = fit_flat_regressor(ridge, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "Ridge", res["y_true"], res["y_pred"])
    all_models[H]["ridge"] = res["model"]
    print("Ridge done")

    huber_ng = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, max_iter=1000))
    ])
    res = fit_flat_regressor(huber_ng, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "Huber", res["y_true"], res["y_pred"])
    all_models[H]["huber"] = res["model"]
    print("No-Graph Huber done")

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    res = fit_flat_regressor(rf, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "RF", res["y_true"], res["y_pred"])
    all_models[H]["rf"] = res["model"]
    print("RF done")

    xgb_direct = XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.8,
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    res = fit_flat_regressor(xgb_direct, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test)
    add_result_row(results_rows, tail_rows, H, "test", "XGB", res["y_true"], res["y_pred"])
    all_models[H]["xgb"] = res["model"]
    print("Direct XGB done")

    # -------------------------
    # ABLATIONS: static graph
    # -------------------------
    ridge_static = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0))
    ])
    res = fit_static_graph_regressor(
        ridge_static,
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        A_static=A_static, tau=best_tau_static
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Static-Graph Ridge",
        res["y_true"], res["y_pred"], extra={"tau": best_tau_static}
    )
    all_models[H]["static_graph_ridge"] = res["model"]
    print("Static-Graph Ridge done")

    huber_static = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, max_iter=1000))
    ])
    res = fit_static_graph_regressor(
        huber_static,
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        A_static=A_static, tau=best_tau_static
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Static-Graph Huber",
        res["y_true"], res["y_pred"], extra={"tau": best_tau_static}
    )
    all_models[H]["static_graph_huber"] = res["model"]
    print("Static-Graph Huber done")

    # -------------------------
    # GRAPH MODEL
    # -------------------------
    graph_plain = fit_huber_graph(
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        ctx=graph_ctx, tau=best_tau_dyn, alpha=4.0, eps=0.05, huber_alpha=1e-4
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Huber Graph",
        graph_plain["y_true"], graph_plain["y_pred"], extra={"tau": best_tau_dyn}
    )
    all_models[H]["graph"] = graph_plain["model"]
    print("Huber Graph done")

    # -------------------------
    # HYBRID MODEL
    # -------------------------
    hybrid_plain = fit_huber_hybrid(
        X_train_imp, Y_train, M_train,
        X_test_imp,  Y_test,  M_test,
        ctx=graph_ctx, tau=best_tau_dyn,
        alpha=4.0, eps=0.05, huber_alpha=1e-4,
        num_boost_round=400, seed=RANDOM_SEED,
        w_mid=2.0, w_danger=5.0, w_tail=10.0
    )
    add_result_row(
        results_rows, tail_rows, H, "test", "Hybrid (Graph + XGB residual)",
        hybrid_plain["y_true"], hybrid_plain["y_pred_final"], extra={"tau": best_tau_dyn}
    )
    all_models[H]["hybrid_graph"] = [hybrid_plain["gmodel"], hybrid_plain["hmodel"]]
    print("Hybrid done")

    # -------------------------
    # TUNED GRAPH + TUNED HYBRID
    # -------------------------
    if RUN_TUNED:
        best_params, best_obj = jso_optimize(
            train_pack=train_pack,
            val_pack=val_pack,
            ctx=graph_ctx,
            tau=best_tau_dyn,
            bounds=bounds,
            n_pop=JSO_POP,
            iters=JSO_ITERS,
            seed=RANDOM_SEED
        )

        alpha, eps, log_huber_alpha, max_depth, eta, subsample, colsample, reg_lambda, nround, w_mid, w_danger, w_tail = best_params
        huber_alpha = 10 ** float(log_huber_alpha)

        tuned_graph = fit_huber_graph(
            X_train_imp, Y_train, M_train,
            X_test_imp,  Y_test,  M_test,
            ctx=graph_ctx, tau=best_tau_dyn,
            alpha=float(alpha), eps=float(eps), huber_alpha=float(huber_alpha)
        )
        add_result_row(
            results_rows, tail_rows, H, "test", "Tuned Huber Graph",
            tuned_graph["y_true"], tuned_graph["y_pred"],
            extra={
                "tau": best_tau_dyn, "alpha": float(alpha), "eps": float(eps),
                "huber_alpha": float(huber_alpha), "val_objective": float(best_obj)
            }
        )
        all_models[H]["tuned_graph"] = tuned_graph["model"]

        tuned_hybrid = fit_huber_hybrid(
            X_train_imp, Y_train, M_train,
            X_test_imp,  Y_test,  M_test,
            ctx=graph_ctx, tau=best_tau_dyn,
            alpha=float(alpha), eps=float(eps), huber_alpha=float(huber_alpha),
            xgb_params={
                "objective": "reg:pseudohubererror",
                "max_depth": int(round(max_depth)),
                "eta": float(eta),
                "subsample": float(subsample),
                "colsample_bytree": float(colsample),
                "lambda": float(reg_lambda),
                "tree_method": "hist",
                "seed": RANDOM_SEED,
                "verbosity": 0,
            },
            num_boost_round=int(round(nround)),
            seed=RANDOM_SEED,
            w_mid=float(w_mid),
            w_danger=float(w_danger),
            w_tail=float(w_tail)
        )
        add_result_row(
            results_rows, tail_rows, H, "test", "Tuned Hybrid (Graph + XGB residual)",
            tuned_hybrid["y_true"], tuned_hybrid["y_pred_final"],
            extra={
                "tau": best_tau_dyn, "alpha": float(alpha), "eps": float(eps),
                "huber_alpha": float(huber_alpha), "max_depth": int(round(max_depth)),
                "eta": float(eta), "subsample": float(subsample),
                "colsample_bytree": float(colsample), "lambda": float(reg_lambda),
                "num_boost_round": int(round(nround)),
                "w_mid": float(w_mid), "w_danger": float(w_danger), "w_tail": float(w_tail),
                "val_objective": float(best_obj)
            }
        )
        all_models[H]["tuned_hybrid_graph"] = [tuned_hybrid["gmodel"], tuned_hybrid["hmodel"]]
        print("Tuned Graph + Tuned Hybrid done")

# =========================================================
# 5) EXCEEDANCE WARNING HEAD
#    Trained only on train, evaluated on test.
#    Threshold = 95th percentile of TRAIN target for that H.
# =========================================================
exceed_rows = []

for H in H_LIST:
    pack = all_data[H]
    X_train_imp = pack["X_train_imp"]
    Y_train = pack["Y_train"]
    M_train = pack["M_train"]
    X_test_imp = pack["X_test_imp"]
    Y_test = pack["Y_test"]
    M_test = pack["M_test"]

    # use the same selected dynamic tau from above
    tau = all_models[H]["selected_tau_dynamic"]

    # graph features -> classifier input
    Ztr = make_graph_features_dynamic(X_train_imp, graph_ctx, tau=tau, alpha=4.0, eps=0.05)
    Zte = make_graph_features_dynamic(X_test_imp, graph_ctx, tau=tau, alpha=4.0, eps=0.05)

    Xc_tr = Ztr.reshape(-1, Ztr.shape[-1])
    Xc_te = Zte.reshape(-1, Zte.shape[-1])

    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = M_train.reshape(-1) > 0.5
    mte = M_test.reshape(-1) > 0.5

    thr = np.percentile(ytr[mtr], 95)
    ytr_cls = (ytr[mtr] >= thr).astype(int)
    yte_cls = (yte[mte] >= thr).astype(int)

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ])
    clf.fit(Xc_tr[mtr], ytr_cls)

    prob = clf.predict_proba(Xc_te[mte])[:, 1]
    pred = (prob >= 0.5).astype(int)

    pr, rc, f1, _ = precision_recall_fscore_support(yte_cls, pred, average="binary", zero_division=0)
    ap = average_precision_score(yte_cls, prob)

    exceed_rows.append({
        "H": H,
        "model": "Graph Exceedance Head (95th pct)",
        "threshold_train_95": float(thr),
        "precision": float(pr),
        "recall": float(rc),
        "f1": float(f1),
        "pr_auc": float(ap),
        "n_test": int(len(yte_cls)),
        "n_positive_test": int(yte_cls.sum())
    })

# =========================================================
# 6) SAVE + DISPLAY
# =========================================================
results_df = pd.DataFrame(results_rows).sort_values(["H", "MAE"]).reset_index(drop=True)
tail_df = pd.DataFrame(tail_rows).sort_values(["H", "model", "percentile"]).reset_index(drop=True)
tau_df = pd.DataFrame(tau_rows).sort_values(["H", "family", "val_MAE"]).reset_index(drop=True)
exceed_df = pd.DataFrame(exceed_rows).sort_values(["H"]).reset_index(drop=True)

results_df.to_csv(os.path.join(OUT_DIR, "all_results_main.csv"), index=False)
tail_df.to_csv(os.path.join(OUT_DIR, "all_results_tail.csv"), index=False)
tau_df.to_csv(os.path.join(OUT_DIR, "selected_tau_validation.csv"), index=False)
exceed_df.to_csv(os.path.join(OUT_DIR, "exceedance_results.csv"), index=False)
joblib.dump(all_models, os.path.join(OUT_DIR, "all_models.pkl"))

print("\nMain results:")
display(results_df)

print("\nTail results (90/95/99 percentiles):")
display(tail_df.head(30))

print("\nValidation tau search:")
display(tau_df)

print("\nExceedance warning results:")
display(exceed_df)

H= 1 | train=17519 | test= 3696 | first train target=2023-01-02 01:00:00 | last train target=2024-12-31 23:00:00 | first test target=2025-04-01 00:00:00
H= 3 | train=17517 | test= 3696 | first train target=2023-01-02 03:00:00 | last train target=2024-12-31 23:00:00 | first test target=2025-04-01 00:00:00
H= 6 | train=17514 | test= 3696 | first train target=2023-01-02 06:00:00 | last train target=2024-12-31 23:00:00 | first test target=2025-04-01 00:00:00
H=12 | train=17508 | test= 3696 | first train target=2023-01-02 12:00:00 | last train target=2024-12-31 23:00:00 | first test target=2025-04-01 00:00:00
H=24 | train=17496 | test= 3696 | first train target=2023-01-03 00:00:00 | last train target=2024-12-31 23:00:00 | first test target=2025-04-01 00:00:00

RUNNING H=1
Selected tau (dynamic) = 6
Selected tau (static)  = 6
Ridge done
No-Graph Huber done
RF done
Direct XGB done
Static-Graph Ridge done
Static-Graph Huber done
Huber Graph done
Hybrid done
    initial best objective=13.4701
 

,H,split,model,MAE,RMSE,R2,n_test,tau,alpha,eps,...,val_objective,max_depth,eta,subsample,colsample_bytree,lambda,num_boost_round,w_mid,w_danger,w_tail
0,1,test,XGB,5.839200,15.540954,0.235267,22992,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,test,Hybrid (Graph + XGB residual),5.959004,17.209178,0.062277,22992,6.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,test,RF,5.965043,16.609605,0.126479,22992,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,test,Huber,6.013383,18.197676,-0.048543,22992,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,test,Tuned Hybrid (Graph + XGB residual),6.152684,18.203147,-0.049174,22992,6.0,3.842952,0.047400,...,13.328727,5.0,0.056233,0.600000,0.847384,4.762315,378.0,3.127499,4.509393,17.005173
5,1,test,Tuned Huber Graph,6.168957,17.925504,-0.017413,22992,6.0,3.842952,0.047400,...,13.328727,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,test,Huber Graph,6.168992,17.925576,-0.017421,22992,6.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,test,Static-Graph Huber,6.169319,17.922256,-0.017044,22992,6.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,test,Ridge,6.811381,18.955594,-0.137704,22992,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,test,Static-Graph Ridge,6.966544,16.720407,0.114786,22992,6.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Tail results (90/95/99 percentiles):


,H,split,model,percentile,threshold,tail_MAE,tail_RMSE,n_tail,tau,alpha,...,val_objective,max_depth,eta,subsample,colsample_bytree,lambda,num_boost_round,w_mid,w_danger,w_tail
0,1,test,Huber,90,37.669998,15.529101,44.145632,2301,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,test,Huber,95,44.984493,21.730429,60.507116,1150,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,test,Huber,99,61.665390,50.391718,130.624832,230,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,test,Huber Graph,90,37.669998,15.707529,43.867398,2301,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,test,Huber Graph,95,44.984493,21.888953,60.677721,1150,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,test,Huber Graph,99,61.665390,49.927055,130.768369,230,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,test,Hybrid (Graph + XGB residual),90,37.669998,15.117895,43.847770,2301,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,test,Hybrid (Graph + XGB residual),95,44.984493,20.708423,60.221677,1150,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,test,Hybrid (Graph + XGB residual),99,61.665390,47.290175,130.072057,230,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,test,RF,90,37.669998,15.298399,43.860437,2301,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Validation tau search:


,tau,val_MAE,H,family
0,6,9.137805,1,dynamic
1,4,9.160159,1,dynamic
2,1,9.160857,1,dynamic
3,2,9.161294,1,dynamic
4,3,9.162245,1,dynamic
5,0,9.172100,1,dynamic
6,6,9.145729,1,static
7,4,9.166560,1,static
8,3,9.170212,1,static
9,2,9.170428,1,static



Exceedance warning results:


,H,model,threshold_train_95,precision,recall,f1,pr_auc,n_test,n_positive_test
0,1,Graph Exceedance Head (95th pct),72.412247,0.083019,0.44,0.139683,0.087990,22992,100
1,3,Graph Exceedance Head (95th pct),72.417007,0.034549,0.18,0.057971,0.017839,22992,100
2,6,Graph Exceedance Head (95th pct),72.416245,0.010118,0.06,0.017316,0.005590,22992,100
3,12,Graph Exceedance Head (95th pct),72.392967,0.000000,0.00,0.000000,0.003763,22992,100
4,24,Graph Exceedance Head (95th pct),72.410492,0.045918,0.18,0.073171,0.025225,22992,100
